# Failytics Risk Engine — Exploratory Data Analysis & Baseline Model

---

## Problem Statement

Distributed services routinely surface failure signals in their telemetry long before a hard outage materialises. The challenge is that operators spend most of their time reacting rather than anticipating — alerts fire after error rates spike, not before. The goal of this project is to turn that telemetry stream into a quantified, short-term failure risk score.

Concretely: given a snapshot of Prometheus metrics and service logs scraped from a live OAuth2 microservice, can a model distinguish a healthy execution path from one that is about to fail — and if so, which signals carry the most predictive weight?

The dataset comes from **light-oauth2**, a production-grade OAuth2 server written in Java, exercised across 53 distinct failure scenarios (malformed requests, invalid credentials, PKCE violations, missing resources) plus a healthy baseline (`correct`). Each row in the dataset is a labelled telemetry snapshot, making this a supervised binary classification problem.

---

## Data Sources

| Source | Path | Description |
|--------|------|-------------|
| **Metrics CSVs** | `data/lo2-sample/metrics/` | 100 Prometheus scrape files. Each covers one test run (~172 rows × 1,128 columns). Columns include Go runtime metrics (`go_memstats_*`, `go_gc_duration_seconds`), OS-level node metrics (`node_cpu_*`, `node_memory_*`, `node_disk_*`), process metrics (`process_cpu_seconds_total`, `process_resident_memory_bytes`), and HTTP handler counters. |
| **Service Logs** | `data/lo2-sample/logs/` | Structured Java application logs from 7 light-oauth2 microservices (token, code, service, client, user, refresh-token, key). One folder per run × scenario. Each log contains timestamped lines with level (DEBUG/INFO/WARN/ERROR), thread, correlation ID, logger class, and message — including full JSON-structured error payloads and stack traces. |

Combined across all files: **17,200 rows** from metrics. Labels: 1,300 healthy (7.6%) and 15,900 failure (92.4%). Log corpus: **37,800 files, ~16 GB**.

---

## Approach

1. Load and consolidate all 100 per-run metric CSVs
2. Clean the data — drop all-NaN and zero-variance columns, remove duplicates, impute remaining gaps
3. Explore class distribution, metric distributions, outliers, and correlations
4. Parse service logs — extract log-level counts, error code frequencies, and exception counts per scenario
5. Analyse log signals — compare healthy vs failure in terms of ERROR rate and error code patterns
6. Engineer features from both sources: unit-normalised metrics, ratio features, log-derived counts, rolling-window statistics
7. Train a Logistic Regression baseline on the combined feature set and evaluate with ROC-AUC


---
## 1. Imports and Setup

In [ ]:
import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    average_precision_score,
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110

METRICS_DIR  = Path('data') / 'lo2-sample' / 'metrics'
LOGS_DIR     = Path('data') / 'lo2-sample' / 'logs'
HEALTHY      = 'correct'
RANDOM_STATE = 42

---
## 2. Data Loading — Metrics

In [ ]:
csv_files = sorted(METRICS_DIR.glob('*.csv'))
print(f'Metric files found: {len(csv_files)}')

frames = [pd.read_csv(f, low_memory=False) for f in csv_files]
df_raw = pd.concat(frames, ignore_index=True)
print(f'Combined shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns')

In [ ]:
# Quick sanity check on the key identifier columns
df_raw[['timestamp', 'run_end', 'test_name']].head()

---
## 3. Data Cleaning

In [ ]:
df = df_raw.copy()

# Convert Unix epoch to timezone-aware datetime
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Binary label: 0 = healthy, 1 = any failure scenario
df['is_failure'] = (df['test_name'] != HEALTHY).astype(int)

# --- Drop all-NaN columns (metrics registered by Prometheus but never populated) ---
before = df.shape[1]
df = df.dropna(axis=1, how='all')
print(f'Dropped {before - df.shape[1]} all-NaN columns')

# --- Drop zero-variance columns (constant across the entire dataset, zero signal) ---
numeric_cols = df.select_dtypes(include='number').columns
zero_var = [c for c in numeric_cols if df[c].nunique() <= 1]
df = df.drop(columns=zero_var)
print(f'Dropped {len(zero_var)} zero-variance columns')

# --- Remove exact duplicate rows ---
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} duplicate rows')

print(f'\nShape after cleaning: {df.shape[0]:,} rows × {df.shape[1]:,} columns')

In [ ]:
# Summarise any remaining missing values before imputation
missing = df.isnull().sum()
missing_summary = (
    pd.DataFrame({'missing_count': missing, 'pct': (missing / len(df) * 100).round(2)})
    .query('missing_count > 0')
    .sort_values('pct', ascending=False)
)

if missing_summary.empty:
    print('No missing values remaining after column drops.')
else:
    print(f'{len(missing_summary)} columns still have missing values:')
    display(missing_summary.head(15))

    # Fill with column median — robust to the outliers that define failure events
    num_cols = df.select_dtypes(include='number').columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    print(f'Imputed with column medians. Missing after: {df.isnull().sum().sum()}')

---
## 4. Exploratory Data Analysis

### 4.1 Dataset Overview

In [ ]:
print('=== Dataset Summary ===')
print(f'  Rows              : {df.shape[0]:,}')
print(f'  Columns           : {df.shape[1]:,}')
print(f'  Time range        : {df["timestamp"].min()} → {df["timestamp"].max()}')
print(f'  Unique test runs  : {df["run_end"].nunique()}')
print(f'  Test scenarios    : {df["test_name"].nunique()}')
print(f'  Healthy rows      : {(df["is_failure"]==0).sum():,}')
print(f'  Failure rows      : {(df["is_failure"]==1).sum():,}')

In [ ]:
# Column groups used throughout the analysis
GC_COLS = [
    'go_gc_duration_seconds&quantile=0',
    'go_gc_duration_seconds&quantile=0.25',
    'go_gc_duration_seconds&quantile=0.5',
    'go_gc_duration_seconds&quantile=0.75',
    'go_gc_duration_seconds&quantile=1',
]
RESOURCE_COLS = [
    'go_goroutines',
    'go_memstats_heap_alloc_bytes',
    'go_memstats_heap_inuse_bytes',
    'go_memstats_heap_sys_bytes',
    'process_resident_memory_bytes',
    'process_cpu_seconds_total',
    'process_open_fds',
]
HTTP_COLS = [
    'promhttp_metric_handler_requests_total&code=200',
    'promhttp_metric_handler_requests_total&code=500',
    'promhttp_metric_handler_requests_in_flight',
]

GC_COLS       = [c for c in GC_COLS       if c in df.columns]
RESOURCE_COLS = [c for c in RESOURCE_COLS if c in df.columns]
HTTP_COLS     = [c for c in HTTP_COLS     if c in df.columns]
KEY_COLS      = GC_COLS + RESOURCE_COLS + HTTP_COLS

display(df[KEY_COLS].describe().T[['mean', 'std', 'min', '50%', 'max']].round(4))

### 4.2 Class Distribution and Imbalance

The dataset is heavily skewed toward failure scenarios — this is by design, since the test harness ran every failure path. With 92% of rows labelled as failures, accuracy is a misleading metric for any model trained here. The evaluation metric choice in the modelling section is driven directly by this imbalance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Binary healthy vs failure ---
class_counts = df['is_failure'].value_counts().sort_index()
bars = axes[0].bar(
    ['Healthy\n(correct)', 'Failure'], class_counts.values,
    color=['#2ecc71', '#e74c3c'], width=0.45, edgecolor='white'
)
for bar, val in zip(bars, class_counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
        f'{val:,}  ({val/len(df)*100:.1f}%)', ha='center', fontsize=10, fontweight='bold'
    )
axes[0].set_title('Binary Class Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Sample Count')
axes[0].set_ylim(0, class_counts.max() * 1.18)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# --- Top 20 failure types ---
top20 = df.query('is_failure == 1')['test_name'].value_counts().head(20)
top20.plot(kind='barh', ax=axes[1], color='#e74c3c', alpha=0.82)
axes[1].set_title('Top 20 Failure Types by Frequency', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sample Count')
axes[1].set_ylabel('')
axes[1].invert_yaxis()
for i, v in enumerate(top20.values):
    axes[1].text(v + 2, i, str(v), va='center', fontsize=8)

plt.tight_layout()
plt.show()

print(f'Imbalance ratio — failures per healthy sample: {class_counts[1] / class_counts[0]:.1f}:1')

### 4.3 Metric Distributions: Healthy vs Failure

KDE plots overlay each metric's distribution for healthy vs failure observations. Separation between the two curves indicates discriminative value.

In [ ]:
PLOT_METRICS = [
    ('go_gc_duration_seconds&quantile=0.5', 'GC Latency p50 (s)'),
    ('go_gc_duration_seconds&quantile=1',   'GC Latency p100 (s)'),
    ('go_goroutines',                       'Goroutine Count'),
    ('go_memstats_heap_alloc_bytes',        'Heap Allocated (bytes)'),
    ('go_memstats_heap_inuse_bytes',        'Heap In-Use (bytes)'),
    ('process_resident_memory_bytes',       'RSS Memory (bytes)'),
    ('process_cpu_seconds_total',           'Process CPU Time (s)'),
    ('process_open_fds',                    'Open File Descriptors'),
    ('promhttp_metric_handler_requests_total&code=200', 'HTTP 200 Requests'),
]
PLOT_METRICS = [(c, lbl) for c, lbl in PLOT_METRICS if c in df.columns]

ncols = 3
nrows = -(-len(PLOT_METRICS) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 4))
axes = axes.flatten()

cls_palette = {0: '#2ecc71', 1: '#e74c3c'}
cls_labels  = {0: 'Healthy', 1: 'Failure'}

for i, (col, title) in enumerate(PLOT_METRICS):
    ax = axes[i]
    for cls, color in cls_palette.items():
        data = df.loc[df['is_failure'] == cls, col].dropna()
        sns.kdeplot(data, ax=ax, label=cls_labels[cls], color=color,
                    fill=True, alpha=0.22, linewidth=1.6)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _:
            f'{x/1e6:.0f}M' if abs(x) >= 1e6 else
            f'{x/1e3:.0f}K' if abs(x) >= 1e3 else f'{x:.3g}'
        )
    )

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Metric Distributions — Healthy vs Failure', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.4 Outlier Analysis

IQR fencing flags observations that lie more than 1.5× the interquartile range outside Q1/Q3. In failure telemetry, these outliers often *are* the signal — a GC pause at 10× the normal p100 is inherently anomalous.

In [ ]:
def iqr_fence(series):
    """Returns IQR bounds and outlier count for a numeric series."""
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((series < lo) | (series > hi)).sum()
    return pd.Series({
        'Q1': q1, 'Q3': q3, 'IQR': iqr,
        'lower_fence': lo, 'upper_fence': hi,
        'outlier_count': int(n_out),
        'outlier_pct': round(n_out / len(series) * 100, 2)
    })

outlier_report = (
    df[GC_COLS + RESOURCE_COLS]
    .apply(iqr_fence)
    .T
    .sort_values('outlier_pct', ascending=False)
    .round(4)
)
display(outlier_report)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# GC latency outlier boxplot
gc_short = {c: 'p' + c.split('quantile=')[1] for c in GC_COLS}
gc_melt = (
    df[GC_COLS + ['is_failure']]
    .rename(columns=gc_short)
    .assign(Class=lambda d: d['is_failure'].map({0: 'Healthy', 1: 'Failure'}))
    .drop(columns='is_failure')
    .melt(id_vars='Class', var_name='Quantile', value_name='gc_duration_s')
)
sns.boxplot(
    data=gc_melt, x='Quantile', y='gc_duration_s', hue='Class', ax=axes[0],
    palette={'Healthy': '#2ecc71', 'Failure': '#e74c3c'},
    flierprops=dict(marker='.', markersize=3, alpha=0.25)
)
axes[0].set_title('GC Duration by Quantile', fontsize=12, fontweight='bold')
axes[0].set_xlabel('GC Latency Quantile')
axes[0].set_ylabel('Duration (s)')

# Resource utilisation outlier boxplot (memory → MB)
res_cols = [c for c in RESOURCE_COLS if c in df.columns]
res_data = df[res_cols + ['is_failure']].copy()
for c in res_cols:
    if 'bytes' in c:
        res_data[c] = res_data[c] / 1e6

short = {
    c: c.replace('go_memstats_', '').replace('process_', '')
         .replace('_bytes', '').replace('_seconds_total', '_s')
    for c in res_cols
}
res_melt = (
    res_data.rename(columns=short)
    .assign(Class=lambda d: d['is_failure'].map({0: 'Healthy', 1: 'Failure'}))
    .drop(columns='is_failure')
    .melt(id_vars='Class', var_name='Metric', value_name='Value')
)
sns.boxplot(
    data=res_melt, x='Metric', y='Value', hue='Class', ax=axes[1],
    palette={'Healthy': '#2ecc71', 'Failure': '#e74c3c'},
    flierprops=dict(marker='.', markersize=3, alpha=0.25)
)
axes[1].set_title('Resource Utilisation (memory = MB, CPU = s)', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()

### 4.5 Correlation Analysis

In [ ]:
corr_cols   = [c for c in KEY_COLS if c in df.columns] + ['is_failure']
corr_matrix = df[corr_cols].corr()

def shorten(col):
    return (
        col.replace('go_gc_duration_seconds&quantile=', 'gc_p')
           .replace('go_memstats_', 'mem.')
           .replace('process_', 'proc.')
           .replace('promhttp_metric_handler_requests_total&code=', 'http_')
           .replace('promhttp_metric_handler_requests_in_flight', 'http_inflight')
           .replace('_bytes', '').replace('_seconds_total', '_s')
    )

short_labels = [shorten(c) for c in corr_cols]
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.4, linecolor='white', annot_kws={'size': 7}, ax=ax
)
ax.set_xticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(short_labels)))
ax.set_yticklabels(short_labels, rotation=0, fontsize=8)
ax.set_title('Correlation Matrix — Key Telemetry Features vs Failure Label',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

top_corr = (
    corr_matrix['is_failure'].drop('is_failure')
    .abs().sort_values(ascending=False)
    .head(10).to_frame('|r|').round(4)
)
print('Top 10 features by absolute correlation with is_failure:')
display(top_corr)

### 4.6 Time-Series View (Interactive)

In [ ]:
ts_agg = (
    df.sort_values('timestamp')
    .groupby('timestamp', as_index=False)
    .agg(
        heap_alloc_mb = ('go_memstats_heap_alloc_bytes',        lambda s: s.mean() / 1e6),
        goroutines    = ('go_goroutines',                       'mean'),
        gc_p50_ms     = ('go_gc_duration_seconds&quantile=0.5', lambda s: s.mean() * 1000),
        failure_rate  = ('is_failure',                          'mean'),
    )
)

fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True,
    subplot_titles=['Heap Allocation (MB)', 'Goroutine Count',
                    'GC Latency p50 (ms)', 'Failure Rate'],
    vertical_spacing=0.06
)
for row, (col, color) in enumerate(
    [('heap_alloc_mb', 'steelblue'), ('goroutines', 'darkorange'),
     ('gc_p50_ms', 'purple'), ('failure_rate', 'crimson')], start=1
):
    fig.add_trace(
        go.Scatter(x=ts_agg['timestamp'], y=ts_agg[col],
                   mode='lines', line=dict(color=color, width=1.5), showlegend=False),
        row=row, col=1
    )

fig.update_layout(height=700, title_text='Telemetry Time-Series Overview',
                  title_font_size=14, template='plotly_white')
fig.show()

### 4.7 Log Analysis

Each test run produces 7 Java service log files per scenario (token, code, service, client, user, refresh-token, key). The logs contain structured per-request entries at DEBUG/INFO/WARN/ERROR level, including full JSON error payloads with error codes (`ERR12014`, `ERR10010`, etc.) and Java exception stack traces.

The parsing step below reads all 37,800 log files and builds a summary dataframe at the `(run_end, test_name)` level — the same granularity as the metrics data, so they can be joined. Each row records: total line count, per-level line counts, error code frequencies, and exception count.

> **Note:** parsing takes approximately 1–2 minutes.

In [ ]:
# Top error codes identified across the dataset (manually verified from one full run)
TOP_ERR_CODES = [
    'ERR12029', 'ERR10010', 'ERR11000', 'ERR11004',
    'ERR12014', 'ERR12015', 'ERR12040', 'ERR12013',
    'ERR12021', 'ERR12039',
]


def parse_scenario_logs(scenario_dir: Path) -> dict:
    """
    Reads all *.log files in a scenario directory and extracts:
      - line-level counts by log level (DEBUG / INFO / WARN / ERROR)
      - structured error code frequencies
      - exception occurrence count
    Returns a flat dict suitable for a single dataframe row.
    """
    total = error = info = debug = warn = exceptions = 0
    code_counts = {c: 0 for c in TOP_ERR_CODES}

    for log_file in scenario_dir.glob('*.log'):
        text = log_file.read_text(errors='replace')
        total      += text.count('\n')
        error      += text.count(' ERROR ')
        info       += text.count(' INFO  ')
        debug      += text.count(' DEBUG ')
        warn       += text.count(' WARN  ')
        exceptions += len(re.findall(r'\bException\b', text))
        for code in re.findall(r'"code":"(ERR\d+)"', text):
            if code in code_counts:
                code_counts[code] += 1

    return {
        'log_lines'          : total,
        'log_error_count'    : error,
        'log_info_count'     : info,
        'log_debug_count'    : debug,
        'log_warn_count'     : warn,
        'log_error_rate'     : error / total if total > 0 else 0,
        'log_exceptions'     : exceptions,
        'log_unique_errcodes': len([v for v in code_counts.values() if v > 0]),
        **{f'log_{k.lower()}': v for k, v in code_counts.items()},
    }


# Cache path — parsing 37,800 files takes ~1-2 min on first run;
# subsequent runs load from parquet in under a second.
LOG_CACHE = Path('data') / 'lo2-sample' / 'log_features.parquet'

if LOG_CACHE.exists():
    log_df = pd.read_parquet(LOG_CACHE)
    print(f'Loaded log features from cache: {log_df.shape}')
else:
    print('Cache not found — parsing log files (this takes ~1-2 min)...')
    log_rows = []
    for run_dir in sorted(LOGS_DIR.iterdir()):
        if not run_dir.is_dir():
            continue
        run_end = run_dir.name
        for scenario_dir in run_dir.iterdir():
            if not scenario_dir.is_dir():
                continue
            row = parse_scenario_logs(scenario_dir)
            row['run_end']   = run_end
            row['test_name'] = scenario_dir.name
            log_rows.append(row)

    log_df = pd.DataFrame(log_rows)
    log_df.to_parquet(LOG_CACHE, index=False)
    print(f'Parsed and cached to {LOG_CACHE}')

log_df['is_failure'] = (log_df['test_name'] != HEALTHY).astype(int)

print(f'Log feature matrix: {log_df.shape[0]:,} rows × {log_df.shape[1]} columns')
print(f'Unique runs: {log_df["run_end"].nunique()}  |  Unique scenarios: {log_df["test_name"].nunique()}')
log_df.drop(columns=['run_end', 'test_name', 'is_failure']).describe().T[['mean', 'std', 'min', '50%', 'max']].round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Log error rate by class ---
sns.violinplot(
    data=log_df, x='is_failure', y='log_error_rate',
    palette={0: '#2ecc71', 1: '#e74c3c'}, inner='quartile', ax=axes[0]
)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Healthy', 'Failure'])
axes[0].set_title('Log ERROR Rate by Class', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('ERROR lines / Total lines')

# --- Log total lines by class ---
sns.boxplot(
    data=log_df, x='is_failure', y='log_lines',
    palette={0: '#2ecc71', 1: '#e74c3c'}, ax=axes[1],
    flierprops=dict(marker='.', markersize=3, alpha=0.3)
)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Healthy', 'Failure'])
axes[1].set_title('Log Volume (Total Lines) by Class', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Total Log Lines')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# --- Exception count by class ---
sns.boxplot(
    data=log_df, x='is_failure', y='log_exceptions',
    palette={0: '#2ecc71', 1: '#e74c3c'}, ax=axes[2],
    flierprops=dict(marker='.', markersize=3, alpha=0.3)
)
axes[2].set_xticks([0, 1])
axes[2].set_xticklabels(['Healthy', 'Failure'])
axes[2].set_title('Exception Count by Class', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Class')
axes[2].set_ylabel('Exception Occurrences')

plt.suptitle('Log-Derived Signals — Healthy vs Failure', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Error code frequency heatmap — top 20 failure scenarios × top 10 error codes
err_code_cols = [c for c in log_df.columns if c.startswith('log_err')]

# Aggregate total count per (test_name, error_code), normalise row-wise
pivot_data = (
    log_df.groupby('test_name')[err_code_cols].sum()
)
# Keep only the top 20 scenarios by total error volume
pivot_data = pivot_data.loc[pivot_data.sum(axis=1).nlargest(20).index]
# Normalise each row so we see the error code *mix*, not just raw counts
row_sums = pivot_data.sum(axis=1).replace(0, 1)
pivot_norm = pivot_data.div(row_sums, axis=0)

# Clean up column labels
pivot_norm.columns = [c.replace('log_', '').upper() for c in pivot_norm.columns]

fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(
    pivot_norm, annot=True, fmt='.2f',
    cmap='YlOrRd', linewidths=0.4, linecolor='white',
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Error Code Mix by Failure Scenario (row-normalised fraction)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Error Code')
ax.set_ylabel('Test Scenario')
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Mean error rate per scenario, ranked — shows which failure types are noisiest in logs
scenario_err = (
    log_df.groupby('test_name')['log_error_rate']
    .mean()
    .sort_values(ascending=True)
)

colors = ['#2ecc71' if t == HEALTHY else '#e74c3c' for t in scenario_err.index]

fig, ax = plt.subplots(figsize=(10, 14))
ax.barh(scenario_err.index, scenario_err.values, color=colors, edgecolor='white')
ax.set_xlabel('Mean Log ERROR Rate (ERROR lines / total lines)')
ax.set_title('Mean Log ERROR Rate per Scenario\n(green = healthy)', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
plt.tight_layout()
plt.show()

---
## 5. Feature Engineering

Features are built from two sources and then joined into a single model-ready matrix:

**Metric features** — derived from raw Prometheus counters:
- Unit-normalised scalars (bytes → MB, seconds → ms)
- Ratio features (`heap_util_pct`, `http_error_rate`) that are run-comparable
- Rolling-window statistics (mean, std, rate-of-change) to capture trajectory over time

**Log features** — extracted per `(run_end, test_name)` from service logs:
- Log level counts and error rate
- Exception count
- Per-error-code occurrence counts
- These are joined back to the metric rows so each row carries both signal types

In [ ]:
feat = df[['timestamp', 'run_end', 'test_name', 'is_failure']].copy()

# --- Metric: unit conversions ---
feat['heap_alloc_mb']  = df['go_memstats_heap_alloc_bytes']  / 1e6
feat['heap_inuse_mb']  = df['go_memstats_heap_inuse_bytes']  / 1e6
feat['rss_mb']         = df['process_resident_memory_bytes'] / 1e6
feat['gc_p50_ms']      = df['go_gc_duration_seconds&quantile=0.5'] * 1000
feat['gc_p100_ms']     = df['go_gc_duration_seconds&quantile=1']   * 1000
feat['goroutines']     = df['go_goroutines']
feat['cpu_s']          = df['process_cpu_seconds_total']
feat['open_fds']       = df['process_open_fds']
feat['gc_count']       = df['go_gc_duration_seconds_count']

# --- Metric: ratio features ---
heap_sys = df['go_memstats_heap_sys_bytes'].replace(0, np.nan)
feat['heap_util_pct']   = (df['go_memstats_heap_alloc_bytes'] / heap_sys * 100).fillna(0)

http_ok  = df['promhttp_metric_handler_requests_total&code=200']
http_err = df['promhttp_metric_handler_requests_total&code=500']
feat['http_error_rate'] = http_err / (http_ok + http_err + 1)

print(f'Metric features built: {feat.shape[1] - 4} feature columns')

In [ ]:
# --- Join log features ---
# log_df is at (run_end, test_name) level; each metric row inherits the log features
# for its matching run × scenario pair
LOG_FEATURE_COLS = [
    'log_error_rate', 'log_error_count', 'log_lines',
    'log_exceptions', 'log_unique_errcodes',
] + [f'log_{c.lower()}' for c in TOP_ERR_CODES]

log_features = log_df[['run_end', 'test_name'] + LOG_FEATURE_COLS]

feat = feat.merge(log_features, on=['run_end', 'test_name'], how='left')

# A small number of rows may not have a matching log dir; fill with 0
feat[LOG_FEATURE_COLS] = feat[LOG_FEATURE_COLS].fillna(0)

print(f'After log join — shape: {feat.shape[0]:,} rows × {feat.shape[1]} columns')
print(f'Unmatched rows: {feat["log_lines"].eq(0).sum()}')

In [ ]:
# --- Rolling-window features on metric signals ---
# Grouped by run_end to prevent bleed across separate test runs
WINDOW_COLS  = ['heap_alloc_mb', 'gc_p50_ms', 'goroutines', 'rss_mb', 'heap_util_pct']
WINDOW_SIZES = [3, 5, 10]

feat = feat.sort_values(['run_end', 'timestamp']).reset_index(drop=True)

for col in WINDOW_COLS:
    grp = feat.groupby('run_end')[col]
    for w in WINDOW_SIZES:
        feat[f'{col}_rmean{w}'] = grp.transform(lambda s: s.rolling(w, min_periods=1).mean())
        feat[f'{col}_rstd{w}']  = grp.transform(lambda s: s.rolling(w, min_periods=1).std().fillna(0))
        feat[f'{col}_roc{w}']   = grp.transform(lambda s: s.diff(w).fillna(0))

id_cols   = ['timestamp', 'run_end', 'test_name', 'is_failure']
feat_cols = [c for c in feat.columns if c not in id_cols]
print(f'Total engineered features: {len(feat_cols)}')

In [ ]:
# Pairplot of base engineered features — sample for readability
pair_cols = ['heap_alloc_mb', 'gc_p50_ms', 'goroutines',
             'heap_util_pct', 'log_error_rate', 'log_exceptions']

sample = feat[pair_cols + ['is_failure']].sample(n=2000, random_state=RANDOM_STATE)
sample['Class'] = sample['is_failure'].map({0: 'Healthy', 1: 'Failure'})

pair_grid = sns.pairplot(
    sample.drop(columns='is_failure'),
    hue='Class',
    palette={'Healthy': '#2ecc71', 'Failure': '#e74c3c'},
    plot_kws={'alpha': 0.3, 's': 12},
    diag_kind='kde',
    corner=True
)
pair_grid.figure.suptitle('Pairplot — Metric + Log Features (2,000 sample)',
                          fontsize=12, fontweight='bold', y=1.01)
plt.show()

---
## 6. Baseline Model

### 6.1 Evaluation Metric Rationale

The dataset has a **12:1 class imbalance** (failures vs healthy). In this setting:

- **Accuracy** is uninformative — a model that always predicts "failure" scores ~92% without learning anything.
- **ROC-AUC** measures the probability that the model ranks a randomly chosen failure observation *higher* than a randomly chosen healthy one. It is threshold-independent and the primary metric here.
- **Average Precision (PR-AUC)** is reported alongside it. Precision-recall curves directly quantify how often high-confidence predictions are correct — useful when the minority class (healthy) is the one worth correctly identifying.
- **F1-score (macro)** balances precision and recall across both classes at a fixed threshold.

A ROC-AUC of 0.5 means random performance; 1.0 is perfect.

### 6.2 Model Choice

**Logistic Regression** is the baseline because:
1. It outputs calibrated probabilities — directly usable as a *failure risk score*.
2. Coefficients are interpretable, showing which features push toward or away from failure.
3. It trains quickly and converges reliably, setting a clean performance floor for future models.

`class_weight='balanced'` upweights the minority (healthy) class automatically.

In [ ]:
# Combined feature set: base metric features + log features
# Rolling features are excluded here — they require temporal train/test splitting
METRIC_FEATURES = [
    'heap_alloc_mb', 'heap_inuse_mb', 'rss_mb',
    'gc_p50_ms', 'gc_p100_ms', 'gc_count',
    'goroutines', 'cpu_s', 'open_fds',
    'heap_util_pct', 'http_error_rate',
]
MODEL_FEATURES = METRIC_FEATURES + LOG_FEATURE_COLS

X = feat[MODEL_FEATURES].values
y = feat['is_failure'].values

# Stratified split preserves the class ratio in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows')
print(f'Features: {X_train.shape[1]}  (metric: {len(METRIC_FEATURES)}, log: {len(LOG_FEATURE_COLS)})')
print(f'Train failure rate: {y_train.mean():.3f}  |  Test failure rate: {y_test.mean():.3f}')

In [ ]:
# Pipeline: StandardScaler → LogisticRegression
# Scaling is required because metric features (bytes → MB) and log counts are on very different scales
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',   LogisticRegression(
                  max_iter=500,
                  class_weight='balanced',
                  random_state=RANDOM_STATE
              )),
])

# 5-fold stratified CV for a stable AUC estimate before touching the test set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'CV ROC-AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

pipe.fit(X_train, y_train)

### 6.3 Evaluation

In [ ]:
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc  = average_precision_score(y_test, y_prob)

print(f'Test ROC-AUC          : {roc_auc:.4f}')
print(f'Test Average Precision: {pr_auc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Healthy', 'Failure']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=['Healthy', 'Failure']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Logistic Regression', fontsize=12, fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='steelblue', lw=2,
             label=f'Logistic Regression (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random baseline (AUC = 0.50)')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])

plt.tight_layout()
plt.show()

### 6.4 Feature Importance

Standardised Logistic Regression coefficients show the direction and magnitude of each feature's contribution. Log-derived features appear alongside metric features — this lets us directly compare the two signal sources.

In [ ]:
coefs = pipe.named_steps['clf'].coef_[0]
importance_df = (
    pd.DataFrame({'feature': MODEL_FEATURES, 'coefficient': coefs})
    .sort_values('coefficient', key=abs, ascending=True)
)

# Colour: red = pushes toward failure, green = pushes toward healthy
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in importance_df['coefficient']]

# Mark log features for visual distinction
labels = [
    f'[LOG] {f}' if f.startswith('log_') else f
    for f in importance_df['feature']
]

fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(labels, importance_df['coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardised Coefficient', fontsize=11)
ax.set_title(
    'Feature Importance — Logistic Regression\n'
    'red = increases failure risk  |  green = decreases  |  [LOG] = log-derived',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.show()

---
## 7. Summary of Findings

**Data characteristics:**
- 17,200 labelled Prometheus snapshots across 100 test runs and 54 scenarios
- 37,800 service log files (~16 GB) providing complementary per-scenario signals
- Class imbalance: 92.4% failure, 7.6% healthy (ratio ~12:1)

**Metric signals:**
- GC latency (p75/p100) is consistently higher in failure scenarios — GC pressure appears as a leading indicator
- Goroutine count and open file descriptors show wider variance in failure runs
- Heap allocation and RSS memory overlap substantially between classes as absolute values; the utilisation ratio (`heap_util_pct`) is a stronger signal

**Log signals:**
- The log ERROR rate is near-zero for healthy scenarios and distinctly elevated for failure ones — a clean separating signal
- Exception counts follow the same pattern: healthy logs contain very few exceptions, most failure scenarios contain structured Java stack traces
- Error codes are scenario-specific: `ERR12014` (CLIENT_NOT_FOUND) concentrates in `access_token_*` failures; `ERR12021` appears in `register_user_*` failures. This means error code features implicitly encode the failure type, which strengthens the model but also makes them somewhat redundant as individual features once the error rate is already included
- Log volume (total line count) is higher for the healthy scenario, likely because successful flows exercise more code paths than failed ones that short-circuit early

**Baseline model:**
- Logistic Regression with `class_weight='balanced'` trained on 11 metric + 15 log features
- Log ERROR rate and exception count rank among the top positive-coefficient features, confirming they add signal beyond the metrics alone
- The probability output is directly interpretable as a failure risk score

**Next steps:**
- Incorporate rolling-window log features (error rate trend over time, not just point-in-time)
- Apply SMOTE or oversampling on the training split to further improve recall on the healthy class
- Replace Logistic Regression with gradient boosting (XGBoost/LightGBM) to capture non-linear interactions between metric and log signals
- Use time-aware cross-validation (GroupKFold on `run_end`) to prevent data leakage across test runs
